In [1]:
import rasterio
from rasterio import features
import numpy as np
import geopandas as gpd
from shapely.geometry import shape
import os

In [ ]:
# --- Configuration ---
input_raster = r"D:\\USLULC"
output_dir = r"D:\\USLULC\\output"

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

# --- 1. Read and Reclassify ---
with rasterio.open(input_raster) as src:
    nlcd_data = src.read(1)
    transform = src.transform
    crs = src.crs

    # Create an empty array for the reclassified data
    reclass_data = np.full(nlcd_data.shape, 5, dtype=np.int16) # Default to 5 (Others)

    # Apply reclassification logic
    reclass_data[np.isin(nlcd_data, [11, 12, 90, 95])] = 1 # Waterbodies
    reclass_data[np.isin(nlcd_data, [21, 22, 23, 24])] = 2 # Impervious
    reclass_data[np.isin(nlcd_data, [41, 42, 43])] = 3 # Green Space
    reclass_data[np.isin(nlcd_data, [51, 52, 71, 72, 73, 74, 81, 82])] = 4 # Low Vegetation

    # (Optional) Save the reclassified raster to visually check it later
    """
    profile = src.profile
    profile.update(dtype=rasterio.int16, count=1)
    with rasterio.open(f"{output_dir}/reclassified.tif", 'w', **profile) as dst:
        dst.write(reclass_data, 1)
    """

# --- 2. Vectorize (Raster to Polygons) ---
print("Extracting polygons...")
# features.shapes yields (geometry, value) pairs
shapes_generator = features.shapes(reclass_data, transform=transform)

# Filter out "Others" (Class 4) and build geometries
records = []
for geom, val in shapes_generator:
    val = int(val)
    if val in [1, 2, 3, 4]:  # Keep only Water, Impervious, Green Space, Low Vegetation
        records.append({
            'geometry': shape(geom),
            'class_id': val
        })

# --- 3. Convert to GeoDataFrame ---
print("Building GeoDataFrame...")
gdf = gpd.GeoDataFrame(records, crs=crs)

# Map class IDs to string names for easier filtering/mapping
class_mapping = {1: 'Waterbodies', 2: 'Impervious', 3: 'Green_Space', 4: 'Low_Vegetation'}
gdf['class_name'] = gdf['class_id'].map(class_mapping)

# --- 4. Export to Shapefile and GeoJSON ---
for class_id, class_name in class_mapping.items():
    print(f"Exporting {class_name}...")
    
    # Isolate the specific class
    class_gdf = gdf[gdf['class_id'] == class_id]
    
    if class_gdf.empty:
        print(f"No data found for {class_name}, skipping.")
        continue

    # File paths
    shp_path = os.path.join(output_dir, f"{class_name}.shp")
    geojson_path = os.path.join(output_dir, f"{class_name}.geojson")

    # Export to Shapefile
    class_gdf.to_file(shp_path)

    # Export to GeoJSON
    # Note: GeoJSON standard assumes WGS84 (EPSG:4326). 
    # If your web map requires it, project it first:
    class_gdf_wgs84 = class_gdf.to_crs(epsg=4326)
    class_gdf_wgs84.to_file(geojson_path, driver="GeoJSON")

print("Processing complete!")